# MongoDB -> WiredTiger Read Path, Top Down

This notebook is organized as a **story from the real entry point downward**.

The goal is not to simulate every detail of MongoDB or WiredTiger. The goal is to make the architecture legible:

1. start at the real MongoDB `find` entry point
2. show the high-level call chain
3. explain what each layer is responsible for
4. only then build a small Python model of that layer
5. finish by running the same logical read under two different snapshots

The notebook stays close to the real code, but narrows the execution path to the simplest useful teaching path: a collection scan over a WiredTiger-backed collection.


## Part 0: The High-Level Map

If you only remember one thing from this notebook, remember this stack:

### Real MongoDB entry point

`find_cmd::run()`

### High-level read path

`find_cmd::run`  
`-> parseQueryAndBeginOperation`  
`-> CanonicalQuery::make`  
`-> getExecutorFind`  
`-> QueryPlanner::plan`  
`-> choose a physical scan shape`  
`-> PlanExecutor (SBE by default today, classic in the simplified model below)`  
`-> CollectionImpl::getCursor`  
`-> RecordStore::getCursor`  
`-> WiredTigerRecordStoreCursor`  
`-> WiredTigerCursor / WT_CURSOR`  
`-> __curfile_next / __curfile_search`  
`-> __wt_btcur_next / __wt_btcur_search`  
`-> __wt_row_search + __wt_page_in_func`  
`-> __wt_txn_visible` decides which version is visible`

That is the full story in one screen.

The rest of the notebook explains each arrow.

### Key source references

- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:620` `find_cmd::run`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:181` `parseQueryAndBeginOperation`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/canonical_query.cpp:89` `CanonicalQuery::make`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/get_executor.cpp:1353` `getExecutorFind`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/query_planner.cpp:1139` `QueryPlanner::plan`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/shard_role/shard_catalog/collection_impl.cpp:545` `CollectionImpl::getCursor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/record_store.h:539` `RecordStore::getCursor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.cpp:1329` `WiredTigerRecordStore::getCursor`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/cursor/cur_file.c:172` `__curfile_next`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/cursor/cur_file.c:297` `__curfile_search`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_curnext.c:638` `__wt_btcur_next`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_cursor.c:716` `__wt_btcur_search`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/include/txn_inline.h:1335` `__wt_txn_visible`


## Part 1: Here Is The Entry Point

The concrete MongoDB command entry point for a `find` is `find_cmd::run()`.

At this level MongoDB is still handling command semantics, authorization, batching, cursor creation, and command-level options. It is **not** yet doing low-level storage reads.

The important teaching point is that `find_cmd::run()` does not jump straight into WiredTiger. It first builds query-layer objects.

A good way to think about the entry point is:

- `find_cmd::run()` says: "we are executing a find command"
- `CanonicalQuery::make()` says: "here is the normalized query"
- `getExecutorFind()` says: "now choose and build the execution machinery"

### Real references for the entry point

- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:620` `find_cmd::run`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:255` CQ construction inside the find command path
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/canonical_query.cpp:97` `CanonicalQuery::CanonicalQuery`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/get_executor.cpp:1353` `getExecutorFind`

### One crucial architectural split

From here onward, keep two questions separate:

- **Query question**: what logical plan should we run?
- **Transaction visibility question**: which record version is visible in this snapshot?

MongoDB query code mostly answers the first. WiredTiger transaction logic mostly answers the second.


In [ ]:
# References:
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:620 `find_cmd::run`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:1096 `_parseCmdObjectToFindCommandRequest`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:181 `parseQueryAndBeginOperation`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/canonical_query.cpp:89 `CanonicalQuery::make`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/get_executor.cpp:1353 `getExecutorFind`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:590 `batchedExecute`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/plan_executor.cpp:84 `PlanExecutor::getNextBatch`
# This cell models the orchestration that `find_cmd::run()` performs through the first batch.

from dataclasses import dataclass
from typing import Any, Dict, List, Optional

class FindCmd:
    """Top-level model of the `find_cmd` command implementation.

    This mirrors the real ownership structure more closely: `run()` is a method on the command
    implementation, while request parsing and first-batch execution are command-owned helpers.
    """
    ref = "/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:620 `find_cmd::run`"
    parse_ref = "/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:1096 `_parseCmdObjectToFindCommandRequest`"
    batch_ref = "/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:590 `batchedExecute`"

    def __init__(self, request_body: Dict[str, Any]) -> None:
        self.request_body = request_body
        self._cmd_request: Optional["FindCommandRequest"] = None

    def _parse_cmd_object_to_find_command_request(self) -> "FindCommandRequest":
        return FindCommandRequest(
            namespace=self.request_body["find"],
            filter=dict(self.request_body.get("filter", {})),
            batch_size=self.request_body.get("batchSize"),
        )

    def batched_execute(
        self,
        batch_size: int,
        executor: "PlanExecutor",
    ) -> List[Dict[str, Any]]:
        return executor.get_next_batch(batch_size)

    def run(
        self,
        op_ctx: "OperationContext",
        collections: "MultipleCollectionAccessor",
    ) -> List[Dict[str, Any]]:
        if self._cmd_request is None:
            self._cmd_request = self._parse_cmd_object_to_find_command_request()

        acquisition = CollectionOrViewAcquisition(namespace=self._cmd_request.namespace)
        canonical_query = parse_query_and_begin_operation(
            acquisition=acquisition,
            request_body=self.request_body,
            find_command=self._cmd_request,
        )
        executor = get_executor_find(op_ctx, collections, canonical_query)
        batch_size = self._cmd_request.batch_size or 101
        return self.batched_execute(batch_size, executor)


def parse_query_and_begin_operation(
    acquisition: "CollectionOrViewAcquisition",
    request_body: Dict[str, Any],
    find_command: "FindCommandRequest",
) -> "CanonicalQuery":
    """Models the free function `parseQueryAndBeginOperation`.

    In the real server this function fills `CurOp`, builds an expression context, parses the
    find grammar, and returns a canonical query ready for planning.
    """
    del acquisition
    del request_body
    return CanonicalQuery.make(find_command)


@dataclass
class FindCommandRequest:
    """Minimal stand-in for MongoDB's parsed `FindCommandRequest`.

    This is the request shape after command parsing and before canonicalization.
    """
    namespace: str
    filter: Dict[str, Any]
    batch_size: Optional[int] = None


@dataclass
class CollectionOrViewAcquisition:
    """Acquired namespace handle passed through the top-level read setup.

    The real type carries collection/view state obtained during acquisition in `find_cmd::run`.
    """
    namespace: str


entry_request_body = {"find": "demo.users", "filter": {"name": "Alice"}, "batchSize": 101}


## Part 2: The Main Characters Before We Execute Anything

Before we model the entry point, we need the important objects on stage.

Think of them like this:

- `OperationContext`: the per-request container
- `RecoveryUnit`: the storage transaction / snapshot handle attached to the request
- `Collection` and `MultipleCollectionAccessor`: how query code refers to an acquired collection
- `RecordStore`: the storage-engine abstraction for reading raw records

This is the first important boundary in the architecture.

Above this boundary, MongoDB is still thinking in query terms. Below it, MongoDB starts to think in storage terms.

### References

- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/operation_context.h:137` `class OperationContext`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/recovery_unit.h:114` `class RecoveryUnit`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/multiple_collection_accessor.h:48` `class MultipleCollectionAccessor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/record_store.h:309` `class RecordStore`


In [ ]:
# References:
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/operation_context.h:137 `class OperationContext`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/recovery_unit.h:114 `class RecoveryUnit`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/multiple_collection_accessor.h:48 `class MultipleCollectionAccessor`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/record_store.h:79 `struct Record`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/record_store.h:198 `class SeekableRecordCursor`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/record_store.h:309 `class RecordStore`

from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Any, Callable, Dict, List, Optional, Set, Tuple

@dataclass
class OperationContext:
    """Per-request container that carries the active RecoveryUnit through the read path.

    This is the first MongoDB object to inspect when you want to know which storage snapshot a
    query is using.
    """
    recovery_unit: "RecoveryUnit"

    def __post_init__(self) -> None:
        pass


@dataclass
class RecoveryUnit:
    """MongoDB's storage-transaction handle.

    This class represents the snapshot owned by the current operation before we commit to any
    specific storage engine internals.
    """
    active: bool = False
    snapshot_id: int = 0

    def abandon_snapshot(self) -> None:
        self.active = False


@dataclass
class Collection:
    """Minimal acquired collection handle.

    Its role in the story is to hand query execution off to the storage layer through
    `get_cursor()`.
    """
    name: str
    record_store: "RecordStore"

    def get_cursor(self, op_ctx: "OperationContext", forward: bool = True) -> "SeekableRecordCursor":
        return self.record_store.get_cursor(op_ctx, op_ctx.recovery_unit, forward)


class RecordStore:
    """Abstract storage-engine surface for reading collection records.

    Query execution code uses this abstraction so it does not need to know about WiredTiger
    directly.
    """
    def get_cursor(self, op_ctx: "OperationContext", ru: "RecoveryUnit", forward: bool = True) -> "SeekableRecordCursor":
        raise NotImplementedError


class SeekableRecordCursor:
    """Engine-facing cursor interface used by MongoDB stages.

    It is the narrow contract between query execution and the storage engine.
    """
    def next(self) -> Optional["Record"]:
        raise NotImplementedError

    def seek_exact(self, record_id: int) -> Optional["Record"]:
        raise NotImplementedError

    def restore(self, ru: "RecoveryUnit") -> bool:
        return True


@dataclass
class MultipleCollectionAccessor:
    """Small stand-in for MongoDB's acquired-collection bundle.

    For this tutorial we only need the main collection, but the real type exists because some
    queries involve multiple collections or views.
    """
    main_collection: "Collection"

    def getMainCollection(self) -> "Collection":
        return self.main_collection


@dataclass
class Record:
    """A raw storage record.

    In the tutorial this is the unit returned by cursors before higher layers think in terms
    of result batches.
    """
    record_id: int
    document: Dict[str, Any]



## Part 3: From Entry Point To Query Planning

Now we model the **MongoDB-side** part of the story.

The important progression is:

- raw `find` request
- `CanonicalQuery`
- `QueryPlanner`
- `PlanExecutor`

This is where MongoDB decides **how it wants to read**. It still is not doing low-level version visibility yet.

For teaching purposes, I force the physical plan to be a collection scan. That is the simplest path that still lets us study the storage and transaction architecture honestly.

### One subtle point

Modern MongoDB usually runs `find` on the SBE engine. This notebook uses a classic-shaped collection-scan executor because it is easier to teach. The storage boundary is still the same one.

### References

- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/canonical_query.h:72` `class CanonicalQuery`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/canonical_query.cpp:89` `CanonicalQuery::make`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/query_planner.cpp:1139` `QueryPlanner::plan`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/compiler/physical_model/query_solution/query_solution.h:535` `CollectionScanNode`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/get_executor.cpp:1353` `getExecutorFind`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/plan_executor_sbe.h:68` `class PlanExecutorSBE`


In [ ]:
# References:
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/canonical_query.h:72 `class CanonicalQuery`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/canonical_query.cpp:89 `CanonicalQuery::make`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/query_planner.cpp:1139 `QueryPlanner::plan`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/compiler/physical_model/query_solution/query_solution.h:535 `CollectionScanNode`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/exec/classic/plan_stage.h:124 `class PlanStage`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/exec/classic/collection_scan.cpp:243 `CollectionScan::doWork`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/plan_executor.h:104 `class PlanExecutor`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/get_executor.cpp:1353 `getExecutorFind`

def get_executor_find(
    op_ctx: OperationContext,
    collections: MultipleCollectionAccessor,
    canonical_query: "CanonicalQuery",
) -> "PlanExecutor":
    """Top-level model of `getExecutorFind`.

    This function is the query-layer handoff after command parsing. In the real server it
    receives a `CanonicalQuery`, chooses a physical plan, and builds an executor before any
    records are read from WiredTiger.
    """
    plan = QueryPlanner().plan(canonical_query)

    root = CollectionScanStage(op_ctx, collections, plan)
    return PlanExecutor(op_ctx, root)


class PlanExecutor:
    """Driver that pulls results from the root stage one unit of work at a time.

    In the real server this is the object the command layer owns while iterating a query. Its
    architectural role is to turn a planned tree into an actual result stream.
    """
    def __init__(self, op_ctx: OperationContext, root: "PlanStage") -> None:
        self.op_ctx = op_ctx
        self.root = root

    def get_next(self) -> Optional["Record"]:
        while True:
            state, record = self.root.work()
            if state is StageState.ADVANCED:
                return record
            if state is StageState.IS_EOF:
                return None

    def get_next_batch(self, batch_size: int) -> List[Dict[str, Any]]:
        """Small stand-in for `PlanExecutor::getNextBatch`.

        `find_cmd::run` uses this method through `batchedExecute` to fill the first batch.
        """
        results: List[Dict[str, Any]] = []
        while len(results) < batch_size:
            record = self.get_next()
            if record is None:
                return results
            results.append(record.document)
        return results

    def execute_all(self) -> List[Dict[str, Any]]:
        results: List[Dict[str, Any]] = []
        while True:
            record = self.get_next()
            if record is None:
                return results
            results.append(record.document)


class PlanStage:
    """Base class for executable query stages.

    It is here so the tutorial can mirror MongoDB's executor contract: the executor repeatedly
    asks a stage to do a small piece of work and reports whether a record was produced.
    """
    def work(self) -> Tuple["StageState", Optional["Record"]]:
        return self.do_work()

    def do_work(self) -> Tuple["StageState", Optional["Record"]]:
        raise NotImplementedError


class StageState(Enum):
    """Subset of MongoDB stage states needed to narrate one scan-based read path."""
    ADVANCED = auto()
    IS_EOF = auto()
    NEED_TIME = auto()


class CollectionScanStage(PlanStage):
    """Executable stage that turns a scan plan into storage-cursor reads.

    This is the first place where the query chapter touches the storage chapter: the stage opens
    a collection cursor and tests each returned record against the canonical predicate.
    """
    def __init__(self, op_ctx: OperationContext, collections: MultipleCollectionAccessor, plan: "CollectionScanNode") -> None:
        self.op_ctx = op_ctx
        self.collections = collections
        self.plan = plan
        self.cursor: Optional["SeekableRecordCursor"] = None
        self.eof = False

    def do_work(self) -> Tuple["StageState", Optional["Record"]]:
        if self.eof:
            return StageState.IS_EOF, None

        if self.cursor is None:
            coll = self.collections.getMainCollection()
            self.cursor = coll.get_cursor(self.op_ctx, forward=True)

        record = self.cursor.next()
        if record is None:
            self.eof = True
            return StageState.IS_EOF, None

        if self.plan.predicate(record.document):
            return StageState.ADVANCED, record

        return StageState.NEED_TIME, None


class QueryPlanner:
    """Planner that chooses a physical access path for a canonical query.

    The real planner explores many alternatives. This notebook chooses only a collection scan so
    the path from planning to visibility stays small and easy to inspect.
    """
    def plan(self, cq: "CanonicalQuery") -> "CollectionScanNode":
        return CollectionScanNode(namespace=cq.namespace, predicate=cq.predicate)


@dataclass
class CollectionScanNode:
    """Physical plan node representing a collection scan.

    This class exists to make the planning chapter concrete: the planner emits an object that
    says the executor should read records by scanning the collection.
    """
    namespace: str
    predicate: Callable[[Dict[str, Any]], bool]


@dataclass
class CanonicalQuery:
    """Normalized query object created from the raw find command.

    Its job is to freeze the logical meaning of the request into a stable shape the planner and
    executor can share.
    """
    namespace: str
    filter: Dict[str, Any]
    predicate: Callable[[Dict[str, Any]], bool]

    @classmethod
    def make(cls, find_command: "FindCommandRequest") -> "CanonicalQuery":
        def predicate(document: Dict[str, Any]) -> bool:
            return all(document.get(key) == value for key, value in find_command.filter.items())

        return cls(find_command.namespace, dict(find_command.filter), predicate)



## Part 4: The Storage Boundary

At this point the query layer has an executor and a scan stage, but it still has not touched WiredTiger internals.

The next boundary is:

`CollectionImpl::getCursor`  
`-> RecordStore::getCursor`  
`-> WiredTigerRecordStoreCursor`

This boundary matters because this is where MongoDB stops saying "give me results for this query" and starts saying "give me a storage cursor over records in the current snapshot".

This is also where transactions become concrete for the read path. The query layer already owns an `OperationContext` and a `RecoveryUnit`, but the underlying WiredTiger read transaction is still lazy.

### What actually becomes real here?

- the storage session
- the read timestamp / read source
- the snapshot that future cursor reads will use

### References

- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/shard_role/shard_catalog/collection_impl.cpp:545` `CollectionImpl::getCursor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/record_store.h:539` `RecordStore::getCursor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.h:478` `class WiredTigerRecordStoreCursorBase`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.cpp:1329` `WiredTigerRecordStore::getCursor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_recovery_unit.cpp:272` `WiredTigerRecoveryUnit::getSession`


In [ ]:
# References:
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_snapshot_manager.h:47 `class WiredTigerSnapshotManager`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_snapshot_manager.cpp:85 `beginTransactionOnCommittedSnapshot`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_recovery_unit.h:69 `class WiredTigerRecoveryUnit`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_recovery_unit.cpp:272 `WiredTigerRecoveryUnit::getSession`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/include/session.h:119 `struct __wt_session_impl`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/include/txn.h:330 `struct __wt_txn`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/include/txn_inline.h:1944 `__wt_txn_begin`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/include/txn_inline.h:1335 `__wt_txn_visible`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/session/session_api.c:637 `__session_open_cursor_int`

class WiredTigerRecoveryUnit(RecoveryUnit):
    """MongoDB recovery-unit implementation backed by a WT session and WT transaction.

    This is the architectural boundary where an abstract MongoDB snapshot becomes a concrete
    WiredTiger read transaction tied to a session.
    """
    _next_snapshot_id = 1

    def __init__(
        self,
        snapshot_manager: "WiredTigerSnapshotManager",
        requested_read_timestamp: int,
        visible_txn_ids: Set[int],
    ) -> None:
        super().__init__()
        self.snapshot_manager = snapshot_manager
        self.requested_read_timestamp = requested_read_timestamp
        self.visible_txn_ids = set(visible_txn_ids)
        self._session: Optional["WiredTigerSession"] = None

    def _ensure_session(self) -> None:
        if self._session is None:
            self._session = WiredTigerSession(WTSessionImpl())

    def get_session(self) -> "WiredTigerSession":
        self._ensure_session()
        if not self.active:
            self.snapshot_manager.begin_transaction_on_committed_snapshot(
                self._session,
                read_timestamp=self.requested_read_timestamp,
                visible_txn_ids=self.visible_txn_ids,
            )
            self.active = True
            self.snapshot_id = WiredTigerRecoveryUnit._next_snapshot_id
            WiredTigerRecoveryUnit._next_snapshot_id += 1
        return self._session


@dataclass
class WiredTigerSnapshotManager:
    """Policy object that chooses which committed snapshot a reader should open.

    MongoDB owns the decision about read source and timestamp. This class exists to model the
    point where that policy is translated into a WT transaction begin operation.
    """
    committed_read_timestamp: int
    committed_txn_ids: Set[int]

    def begin_transaction_on_committed_snapshot(
        self,
        session: "WiredTigerSession",
        read_timestamp: Optional[int] = None,
        visible_txn_ids: Optional[Set[int]] = None,
    ) -> int:
        chosen_read_ts = self.committed_read_timestamp if read_timestamp is None else read_timestamp
        chosen_visible = self.committed_txn_ids if visible_txn_ids is None else visible_txn_ids
        session.impl.txn.begin(chosen_read_ts, set(chosen_visible))
        return chosen_read_ts


@dataclass
class WiredTigerSession:
    """MongoDB-facing wrapper around `WT_SESSION_IMPL`.

    The wrapper is important because MongoDB code usually talks in terms of WiredTigerSession,
    while the deeper visibility logic consults the embedded WT session implementation.
    
    This is also where the CURSOR FACTORY PATTERN lives in WiredTiger:
    `open_cursor()` routes URIs to different cursor implementations.
    """
    impl: "WTSessionImpl"
    
    def open_cursor(
        self, 
        uri: str, 
        btree: Optional["WTBTree"] = None,
        config: Optional[str] = None
    ) -> "WiredTigerCursor":
        """The WiredTiger cursor factory: routes URI to cursor type.
        
        This mirrors `__session_open_cursor_int` in session_api.c:
        - "table:" -> table cursor
        - "file:"  -> btree file cursor  
        - "index:" -> index cursor
        - etc.
        
        The URI prefix determines which underlying cursor implementation gets created.
        This is how WiredTiger supports pluggable data sources.
        """
        # Simplified URI parsing - in real WT this is a switch statement
        # See /Users/gabrielelanaro/workspace/wiredtiger/src/session/session_api.c:654
        if uri.startswith("table:") or uri.startswith("file:"):
            if btree is None:
                raise ValueError(f"URI {uri} requires btree parameter")
            # In real WiredTiger: __wt_curfile_open -> __wt_btcur_open
            return self._open_btree_cursor(uri, btree)
        elif uri.startswith("index:"):
            if btree is None:
                raise ValueError(f"URI {uri} requires btree parameter")
            # In real WiredTiger: __wt_curindex_open
            return self._open_btree_cursor(uri, btree)
        else:
            raise ValueError(f"Unsupported URI prefix: {uri}")
    
    def _open_btree_cursor(self, uri: str, btree: "WTBTree") -> "WiredTigerCursor":
        """Create a btree cursor - this is what MongoDB uses for collections."""
        return WiredTigerCursor(uri, self, btree)


@dataclass
class WTTxn:
    """Tiny model of `WT_TXN` visibility state.

    This class exists for the central question of the storage chapter: given a version chain,
    which version is visible to the current reader's snapshot?
    """
    read_timestamp: Optional[int] = None
    visible_txn_ids: Set[int] = field(default_factory=set)
    id: Optional[int] = None
    running: bool = False

    def begin(self, read_timestamp: int, visible_txn_ids: Set[int]) -> None:
        self.read_timestamp = read_timestamp
        self.visible_txn_ids = set(visible_txn_ids)
        self.running = True

    def visible(self, version: "VersionedValue") -> bool:
        if not self.running:
            raise RuntimeError("WT transaction is not running")

        if self.id is not None and version.start_txn == self.id:
            return True
        if version.start_txn not in self.visible_txn_ids:
            return False
        if self.read_timestamp is not None and version.start_ts > self.read_timestamp:
            return False

        if version.stop_txn is None:
            return True
        if self.id is not None and version.stop_txn == self.id:
            return False
        if version.stop_txn not in self.visible_txn_ids:
            return True

        stop_ts = version.stop_ts or version.durable_stop_ts
        if stop_ts is None or self.read_timestamp is None:
            return False
        return stop_ts > self.read_timestamp


@dataclass
class WTSessionImpl:
    """Minimal stand-in for `WT_SESSION_IMPL`.

    Its purpose here is to carry the active `WTTxn` so the lower layers can find the reader
    snapshot state in one place.
    
    In real WiredTiger, this struct also contains:
    - Connection reference
    - Cursor cache (for cursor reuse)
    - Transaction state
    - Statistics
    """
    txn: WTTxn = field(default_factory=WTTxn)


@dataclass
class VersionedValue:
    """One logical record version in the MVCC chain.

    This class is the thing `__wt_txn_visible` reasons about: a reader is not choosing a record
    id, it is choosing which committed version of that record id can be seen.
    """
    record_id: int
    document: Dict[str, Any]
    start_txn: int
    start_ts: int
    durable_start_ts: Optional[int] = None
    stop_txn: Optional[int] = None
    stop_ts: Optional[int] = None
    durable_stop_ts: Optional[int] = None


## Part 4.5: The Cursor Factory Pattern - Why It Matters

The cursor factory pattern is **central** to WiredTiger's architecture. It's not just about creating cursors - it's about how WiredTiger supports multiple data sources through a single API.

### The Factory Pattern in Action

When MongoDB calls `session->open_cursor(uri, ...)`, WiredTiger routes based on the URI prefix:

```c
// From /Users/gabrielelanaro/workspace/wiredtiger/src/session/session_api.c:654
switch (uri[0]) {
case 't':
    if (WT_PREFIX_MATCH(uri, "table:"))
        WT_RET(__wt_curtable_open(session, uri, owner, cfg, cursorp));
    break;
case 'f':
    if (WT_PREFIX_MATCH(uri, "file:"))
        WT_RET(__wt_curfile_open(session, uri, owner, cfg, cursorp));
    break;
case 'i':
    if (WT_PREFIX_MATCH(uri, "index:"))
        WT_RET(__wt_curindex_open(session, uri, owner, cfg, cursorp));
    break;
// ... many more cursor types
}
```

### Why This Matters

1. **Pluggable Data Sources**: Anyone can add a new cursor type by registering it with the factory
2. **URI-based Routing**: The same `open_cursor` call works for tables, indexes, logs, statistics, etc.
3. **Configuration Centralization**: Cursor options are parsed once in the factory
4. **Cursor Caching**: The session can cache cursors and reuse them because creation goes through one place

### In Your Tutorial

Now the cursor creation flow matches the real architecture:

```
MongoDB: WiredTigerRecoveryUnit::getSession()
  └─> WiredTigerSession::open_cursor(uri, btree)  ← FACTORY!
       └─> Routes URI to cursor type
            └─> Returns WiredTigerCursor
                 └─> Contains WTCursorBTree (the actual btree operations)
```

This is how MongoDB creates collection cursors: `table:collection.name` → btree cursor.

### References

- `/Users/gabrielelanaro/workspace/wiredtiger/src/session/session_api.c:637` `__session_open_cursor_int`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_session.cpp:102` `WiredTigerSession::open_cursor`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/cursor/cur_file.c:172` `__curfile_open`


In [ ]:
# DEMO: The Cursor Factory Pattern in Action
#
# This cell explicitly shows how open_cursor() works, step by step.
# Compare with the real MongoDB code at:
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_session.cpp:102

# Step 1: Create a recovery unit (this is MongoDB's transaction handle)
demo_ru = WiredTigerRecoveryUnit(
    snapshot_manager=snapshot_manager,
    requested_read_timestamp=60,
    visible_txn_ids={10, 50},
)

# Step 2: Get the session (this creates/lazily initializes the WT session)
print("Step 1: Getting session from recovery unit...")
demo_session = demo_ru.get_session()
print(f"  ✓ Got session with txn running: {demo_session.impl.txn.running}")
print(f"  ✓ Read timestamp: {demo_session.impl.txn.read_timestamp}")

# Step 3: Call open_cursor - THIS IS THE FACTORY PATTERN!
print("\nStep 2: Calling session.open_cursor() - THE CURSOR FACTORY!")
print("  This is the key architectural pattern you asked about.")
print()

# Different URIs create different cursor types (in real WT)
# For this demo, we'll show the table: and index: paths

table_uri = "table:demo.users"
print(f"Creating cursor for URI: '{table_uri}'")
table_cursor = demo_session.open_cursor(uri=table_uri, btree=btree)
print(f"  ✓ Created WiredTigerCursor for table")

# This demonstrates that the same factory method handles different URIs
# In real WiredTiger, this would create different cursor implementations:
#   "table:xxx" → table cursor with column groups
#   "file:xxx"  → btree file cursor
#   "index:xxx" → index cursor

print("\nStep 3: Using the cursor to read data")
record = table_cursor.search(1)
if record:
    print(f"  ✓ Found record: {record.document}")

print("\n" + "="*60)
print("KEY TAKEAWAY:")
print("="*60)
print("MongoDB NEVER directly creates WiredTigerCursor.")
print("It ALWAYS goes through session.open_cursor(uri).")
print("The URI determines which cursor type the factory creates.")
print()
print("This is why the cursor factory pattern is so important!")


## Part 5: The WiredTiger Internal Read Path

Now we are below the MongoDB storage abstraction.

This is the part where people often say "this is the real read path", but by now you should see that it is only the **lower half** of the story.

Here is the simplified internal progression we will model:

`WiredTigerRecordStoreCursor.next()`  
`-> WT cursor next/search`  
`-> btree cursor function`  
`-> locate page / row`  
`-> choose visible version`

The crucial fact is this:

**A read in WiredTiger is not just "find the key". It is "find the newest version of the key visible to this transaction snapshot".**

That is why `__wt_txn_visible()` is the conceptual center of the storage read path.

### Real lower-level path references

- `/Users/gabrielelanaro/workspace/wiredtiger/src/cursor/cur_table.c:260` `__curtable_next`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/cursor/cur_table.c:370` `__curtable_search`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/cursor/cur_file.c:172` `__curfile_next`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/cursor/cur_file.c:297` `__curfile_search`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_curnext.c:638` `__wt_btcur_next`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_cursor.c:716` `__wt_btcur_search`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/row_srch.c:355` `__wt_row_search`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_read.c:469` `__wt_page_in_func`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/include/txn_inline.h:2142` `__wt_txn_search_check`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/include/txn_inline.h:1335` `__wt_txn_visible`


In [ ]:
# References:
# - /Users/gabrielelanaro/workspace/wiredtiger/src/include/cursor.h:116 `struct __wt_cursor_btree`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/include/btree.h:113 `struct __wt_btree`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/include/btmem.h:595 `struct __wt_page`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/include/btmem.h:1149 `struct __wt_ref`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_read.c:469 `__wt_page_in_func`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/btree/row_srch.c:355 `__wt_row_search`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_cursor.c:716 `__wt_btcur_search`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_curnext.c:638 `__wt_btcur_next`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_cursor.h:47 `class WiredTigerCursor`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.cpp:1329 `WiredTigerRecordStore::getCursor`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.cpp:1675 `WiredTigerRecordStoreCursorBase::next`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.cpp:1831 `WiredTigerRecordStoreCursorBase::seekExactCommon`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_session.cpp:102 `WiredTigerSession::open_cursor`

class WiredTigerRecordStoreCursor(SeekableRecordCursor):
    """MongoDB storage cursor that adapts RecordStore reads onto a WT-backed cursor.

    This is the main bridge in the read path. Query execution calls `next()` or `seek_exact()`
    here, and this class translates that request into a WiredTiger cursor operation under the
    active snapshot.
    """
    def __init__(self, op_ctx: "OperationContext", ru: "RecoveryUnit", record_store: "WiredTigerRecordStore", forward: bool) -> None:
        self.op_ctx = op_ctx
        self.ru = ru
        self.record_store = record_store
        self.forward = forward
        self.cursor: Optional["WiredTigerCursor"] = None

    def restore(self, ru: "RecoveryUnit") -> bool:
        """Restore or create cursor using the CURSOR FACTORY PATTERN.
        
        This mirrors the real MongoDB path:
        - WiredTigerRecoveryUnit::getSession() gets the session
        - session->open_cursor() creates the cursor via the factory
        
        See: /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_session.cpp:102
        """
        self.ru = ru
        if not isinstance(ru, WiredTigerRecoveryUnit):
            raise TypeError("Expected a WiredTigerRecoveryUnit")
        session = ru.get_session()
        # CURSOR FACTORY: session.open_cursor() routes URI to cursor type
        self.cursor = session.open_cursor(
            uri=self.record_store.uri,
            btree=self.record_store.btree
        )
        return True

    def next(self) -> Optional["Record"]:
        if self.cursor is None:
            self.restore(self.ru)
        return self.cursor.next()

    def seek_exact(self, record_id: int) -> Optional["Record"]:
        if self.cursor is None:
            self.restore(self.ru)
        return self.cursor.search(record_id)


class WiredTigerRecordStore(RecordStore):
    """WiredTiger-backed implementation of MongoDB's `RecordStore` abstraction.

    The query layer never instantiates WT cursors directly. It asks the record store for a
    cursor, and this class produces a cursor that is already tied to the current recovery unit.
    """
    def __init__(self, uri: str, btree: "WTBTree") -> None:
        self.uri = uri
        self.btree = btree

    def get_cursor(self, op_ctx: "OperationContext", ru: "RecoveryUnit", forward: bool = True) -> "SeekableRecordCursor":
        cursor = WiredTigerRecordStoreCursor(op_ctx, ru, self, forward)
        cursor.restore(ru)
        return cursor


class WiredTigerCursor:
    """MongoDB-side wrapper around a lower-level WT btree cursor.
    
    IMPORTANT: This cursor is CREATED via the CURSOR FACTORY PATTERN:
        session.open_cursor(uri, btree) -> WiredTigerCursor
    
    The factory pattern is crucial because:
    1. Different URI prefixes create different cursor types (table, index, etc.)
    2. Cursor configuration is centralized
    3. Extensibility - new data sources can plug into the factory
    
    This wrapper is present in the real code because MongoDB owns cursor lifetime 
    and error handling before handing individual operations down to WiredTiger.
    """
    def __init__(self, uri: str, session: "WiredTigerSession", btree: "WTBTree") -> None:
        self.uri = uri
        self.session = session
        # The underlying btree cursor (WT_CURSOR_BTREE in real WiredTiger)
        self._cursor = WTCursorBTree(session.impl, btree)

    def next(self) -> Optional["Record"]:
        return self._cursor.next()

    def search(self, record_id: int) -> Optional["Record"]:
        return self._cursor.search(record_id)


class WTCursorBTree:
    """Toy stand-in for the btree cursor logic under `WT_CURSOR`.
    
    In real WiredTiger, this corresponds to:
    - WT_CURSOR_BTREE structure (cursor/cur_file.c)
    - Functions: __wt_btcur_next, __wt_btcur_search
    
    This class makes the lower-level read story visible: before a record is returned, WT checks
    transaction state, touches the btree, and chooses the newest visible version.
    """
    def __init__(self, session: "WTSessionImpl", btree: "WTBTree") -> None:
        self.session = session
        self.btree = btree
        self._scan_results: Optional[List["Record"]] = None
        self._scan_index = 0

    def _txn_search_check(self) -> None:
        """Mirror of __wt_txn_search_check - ensures active transaction."""
        if not self.session.txn.running:
            raise RuntimeError("No active WT read transaction")

    def search(self, record_id: int) -> Optional["Record"]:
        """Search for a specific record by key.
        
        Real path: __wt_btcur_search -> __wt_row_search -> __wt_page_in_func
        """
        self._txn_search_check()
        versions = self.btree.row_search(self.session, record_id)
        version = self.btree.visible_version_for_read(self.session, versions)
        return None if version is None else Record(record_id, version.document)

    def next(self) -> Optional["Record"]:
        """Advance to next record.
        
        Real path: __wt_btcur_next -> __wt_row_search -> __wt_page_in_func
        """
        self._txn_search_check()
        if self._scan_results is None:
            self._scan_results = self.btree.iter_visible_records(self.session)
        if self._scan_index >= len(self._scan_results):
            return None
        record = self._scan_results[self._scan_index]
        self._scan_index += 1
        return record


@dataclass
class WTBTree:
    """Simplified btree container with page lookup and visibility-aware record helpers.

    It is the notebook's small stand-in for the deeper WiredTiger tree code beneath the cursor.
    The purpose is not to emulate the whole engine, only to show where page access and version
    choice happen.
    """
    name: str
    root: "WTRef"

    def page_in(self, session: "WTSessionImpl", ref: "WTRef") -> "WTPage":
        """Mirror of __wt_page_in_func - bring a page into memory."""
        return ref.page

    def row_search(self, session: "WTSessionImpl", record_id: int) -> List["VersionedValue"]:
        """Mirror of __wt_row_search - locate versions for a key."""
        page = self.page_in(session, self.root)
        return page.versions_by_record_id.get(record_id, [])

    def visible_version_for_read(self, session: "WTSessionImpl", versions: List["VersionedValue"]) -> Optional["VersionedValue"]:
        """Apply transaction visibility to choose the correct version.
        
        This mirrors the logic in __wt_txn_visible / __wt_txn_search_check.
        """
        visible_versions = [version for version in versions if session.txn.visible(version)]
        if not visible_versions:
            return None
        return max(visible_versions, key=lambda version: (version.start_ts, version.start_txn))

    def iter_visible_records(self, session: "WTSessionImpl") -> List["Record"]:
        """Scan all records visible in the current transaction snapshot."""
        page = self.page_in(session, self.root)
        out: List["Record"] = []
        for record_id in page.sorted_keys():
            version = self.visible_version_for_read(session, page.versions_by_record_id[record_id])
            if version is not None:
                out.append(Record(record_id, version.document))
        return out


@dataclass
class WTRef:
    """Reference from a btree node to an in-memory page.

    The tutorial includes it because real WiredTiger tree traversal reaches pages through
    `WT_REF` objects rather than storing raw page pointers everywhere.
    """
    page: "WTPage"
    state: str = "mem"


@dataclass
class WTPage:
    """In-memory page containing version chains keyed by record id.

    This is the lowest-level data container in the notebook. It holds the version chains that
    `WTTxn.visible()` filters when a reader walks the tree.
    """
    versions_by_record_id: Dict[int, List["VersionedValue"]]

    def sorted_keys(self) -> List[int]:
        return sorted(self.versions_by_record_id)


## Part 6: Now We Rebuild The Entry Point As A Story

The next cell is the whole notebook compressed into one top-down function.

Read it as a story, not just as code:

1. the command arrives
2. MongoDB creates an `OperationContext`
3. MongoDB calls the query entry point (`getExecutorFind` in the real code)
4. the executor opens a collection cursor
5. the record store opens a WiredTiger-backed cursor
6. WiredTiger opens a read transaction on the requested snapshot
7. the btree returns the newest visible version of each record

After that, we run the same logical request against two snapshots so the transaction semantics become visible.


In [ ]:
# References:
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:620 `find_cmd::run`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/get_executor.cpp:1353 `getExecutorFind`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/shard_role/shard_catalog/collection_impl.cpp:545 `CollectionImpl::getCursor`
# - /Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.cpp:1329 `WiredTigerRecordStore::getCursor`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_curnext.c:638 `__wt_btcur_next`
# - /Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_cursor.c:716 `__wt_btcur_search`

def run_find_request(
    request_body: Dict[str, Any],
    read_ts: int,
    visible_txn_ids: Set[int],
) -> List[Dict[str, Any]]:
    """Run the notebook through the same `FindCmd.run()` entry point shown in Part 1."""
    ru = WiredTigerRecoveryUnit(
        snapshot_manager=snapshot_manager,
        requested_read_timestamp=read_ts,
        visible_txn_ids=visible_txn_ids,
    )
    op_ctx = OperationContext(ru)
    return FindCmd(request_body).run(op_ctx, collections)


def run_seek_exact(
    record_id: int,
    read_ts: int,
    visible_txn_ids: Set[int],
) -> Optional[Dict[str, Any]]:
    """Run the shorter point-lookup story through the storage cursor boundary.

    This companion function is useful because it strips away planning and executor iteration,
    leaving just the storage bridge and the final WiredTiger search path.
    """
    ru = WiredTigerRecoveryUnit(
        snapshot_manager=snapshot_manager,
        requested_read_timestamp=read_ts,
        visible_txn_ids=visible_txn_ids,
    )
    op_ctx = OperationContext(ru)
    cursor = collection.get_cursor(op_ctx, forward=True)
    record = cursor.seek_exact(record_id)
    return None if record is None else record.document


alice_v1 = VersionedValue(
    record_id=1,
    document={"_id": 1, "name": "Alice", "age": 30},
    start_txn=10,
    start_ts=10,
    durable_start_ts=10,
    stop_txn=50,
    stop_ts=50,
    durable_stop_ts=50,
)
alice_v2 = VersionedValue(
    record_id=1,
    document={"_id": 1, "name": "Alice", "age": 31},
    start_txn=50,
    start_ts=50,
    durable_start_ts=50,
)
bob_v1 = VersionedValue(
    record_id=2,
    document={"_id": 2, "name": "Bob", "age": 35},
    start_txn=10,
    start_ts=10,
    durable_start_ts=10,
)

page = WTPage({1: [alice_v1, alice_v2], 2: [bob_v1]})
btree = WTBTree(name="table:demo.users", root=WTRef(page))
snapshot_manager = WiredTigerSnapshotManager(
    committed_read_timestamp=60,
    committed_txn_ids={10, 50},
)
record_store = WiredTigerRecordStore(uri="table:demo.users", btree=btree)
collection = Collection(name=entry_request_body["find"], record_store=record_store)
collections = MultipleCollectionAccessor(main_collection=collection)


old_results = run_find_request(dict(entry_request_body), read_ts=40, visible_txn_ids={10})
new_results = run_find_request(dict(entry_request_body), read_ts=60, visible_txn_ids={10, 50})
seek_result = run_seek_exact(1, read_ts=60, visible_txn_ids={10, 50})

print(f"find({entry_request_body['filter']}) at read_ts=40 ->", old_results)
print(f"\nfind({entry_request_body['filter']}) at read_ts=60 ->", new_results)
print("\nseek_exact(record_id=1) at read_ts=60 ->", seek_result)


## Part 7: What The Two Runs Teach You

The same logical query produced different visible versions because the snapshot changed.

That is the key architectural lesson:

- the **query path** is mostly the same in both runs
- the **visible document version** changes because the storage snapshot changes

So when you study MongoDB + WiredTiger transactions, do not treat the read path as a single undifferentiated pipeline. It is really two interacting pipelines:

### Pipeline A: query execution

`find_cmd::run -> CanonicalQuery -> planner -> executor -> collection scan`

### Pipeline B: snapshot visibility

`RecoveryUnit -> WiredTigerRecoveryUnit -> WT_SESSION_IMPL/WT_TXN -> __wt_txn_visible`

Understanding the architecture means understanding where those two pipelines meet.

They meet at the storage cursor boundary.


## Part 8: Reading Order In The Real Source

If you now want to read the actual code in a sane order, use this sequence.

### First pass: top-down MongoDB path

- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/commands/query_cmd/find_cmd.cpp:620` `find_cmd::run`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/canonical_query.cpp:89` `CanonicalQuery::make`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/get_executor.cpp:1353` `getExecutorFind`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/query_planner.cpp:1139` `QueryPlanner::plan`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/query/compiler/physical_model/query_solution/query_solution.h:535` `CollectionScanNode`

### Second pass: executor to storage boundary

- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/exec/classic/collection_scan.cpp:243` `CollectionScan::doWork`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/shard_role/shard_catalog/collection_impl.cpp:545` `CollectionImpl::getCursor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/record_store.h:539` `RecordStore::getCursor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.cpp:1329` `WiredTigerRecordStore::getCursor`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_record_store.cpp:1675` `WiredTigerRecordStoreCursorBase::next`

### Third pass: snapshot and visibility boundary

- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_recovery_unit.cpp:272` `WiredTigerRecoveryUnit::getSession`
- `/Users/gabrielelanaro/workspace/mongodb/src/mongo/db/storage/wiredtiger/wiredtiger_snapshot_manager.cpp:85` `beginTransactionOnCommittedSnapshot`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/include/txn_inline.h:1944` `__wt_txn_begin`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/include/txn_inline.h:2142` `__wt_txn_search_check`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_cursor.c:716` `__wt_btcur_search`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_curnext.c:638` `__wt_btcur_next`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/row_srch.c:355` `__wt_row_search`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/btree/bt_read.c:469` `__wt_page_in_func`
- `/Users/gabrielelanaro/workspace/wiredtiger/src/include/txn_inline.h:1335` `__wt_txn_visible`

### Final mental model

MongoDB decides **what path to execute**. WiredTiger decides **what version is visible on that path**.


## WiredTiger Class Relationships

Below are key diagrams showing how WiredTiger classes relate to each other:

# WiredTiger Architecture Diagram

## Class Relationship Overview

```mermaid
classDiagram
    WT_CONNECTION_IMPL "-->" "1..*" WT_SESSION_IMPL : creates
    WT_SESSION_IMPL "1" --> "1" WT_TXN : contains
    WT_SESSION_IMPL "1" --> "1..*" WT_CURSOR : creates

    WT_CURSOR "*" --> "1" WT_SESSION_IMPL : session (CUR2S)
    WT_CURSOR_BTREE --|> WT_CURSOR
    WT_CURSOR_BTREE "1" --> "1" WT_BTREE : operates on

    WT_BTREE "*" --> "1" WT_CONNECTION_IMPL : owned by
    WT_BTREE "1" --> "1..*" WT_REF : root/reference
    WT_REF "*" --> "1" WT_PAGE : points to

    WT_PAGE "1" --> "0..*" WT_MODIFY : modifications
    WT_PAGE "1" --> "0..*" WT_UPDATE : update chain

    class WT_CONNECTION_IMPL {
        +WT_SESSION_IMPL* default_session
        +WT_BTREE* btrees
        open_session()
        open_cursor()
    }

    class WT_SESSION_IMPL {
        +WT_TXN* txn
        +WT_CONNECTION_IMPL* connection
        +WT_CURSOR* cursor_cache
        begin_transaction()
        commit_transaction()
        open_cursor(uri, config)
    }

    class WT_TXN {
        +uint64_t id
        +WT_TXN_ISOLATION isolation
        +uint64_t snapshot_min
        +wt_timestamp_t read_timestamp
        +bool running
        visible(txn_id, timestamp)
    }

    class WT_CURSOR {
        +WT_SESSION_IMPL* session
        +WT_ITEM key
        +WT_ITEM value
        <<interface>>
    }

    class WT_CURSOR_BTREE {
        +WT_CURSOR iface
        +WT_REF* ref
        +WT_BTREE* btree
        +int compare
        search(key)
        next()
        reset()
    }

    class WT_BTREE {
        +WT_SESSION_IMPL* session
        +WT_REF* root
        +uint32_t pagemax
        key_format
        value_format
    }

    class WT_REF {
        +WT_PAGE* page
        +wt_timestamp_t rec_max_txn
        +uint8_t state
        +addr_t addr
    }

    class WT_PAGE {
        +WT_PAGE_HEADER* header
        +void* dsk
        +WT_MODIFY* modify
        +u_int type
    }

    class WT_MODIFY {
        +WT_UPDATE* first_update
        +WT_UPDATE* last_update
        +uint32_t write_gen
    }

    class WT_UPDATE {
        +WT_TXN* txn
        +wt_timestamp_t start_ts
        +wt_timestamp_t durable_ts
        +WT_UPDATE* next
        +data/value
    }
```

## Key Navigation Macros

```mermaid
graph LR
    CURSOR[WT_CURSOR_BTREE* cbt] -->|"CUR2S(cbt)"| SESSION[WT_SESSION_IMPL* session]
    SESSION -->|"session->txn"| TXN[WT_TXN* txn]
    SESSION -->|"S2BT(session)"| BTREE[WT_BTREE* btree]

    style CURSOR fill:#e1f5ff
    style SESSION fill:#fff4e1
    style TXN fill:#ffe1e1
    style BTREE fill:#e1ffe1
```

## Cursor Lifecycle Flow

```mermaid
sequenceDiagram
    participant RU as RecoveryUnit
    participant WS as WiredTigerSession
    participant WC as WiredTigerCursor
    participant CUR as WT_CURSOR_BTREE
    participant BT as WT_BTREE
    participant TX as WT_TXN

    RU->>WS: getSession()
    WS->>WS: _ensureSession()
    RU->>WS: open_cursor(uri, btree)

    Note over WS,CUR: CURSOR FACTORY PATTERN
    WS->>CUR: wt_session->open_cursor(uri)
    CUR->>BT: btree = S2BT(session)
    CUR->>CUR: initialize cursor state

    RU->>WC: new WiredTigerCursor(uri, session, btree)
    WC->>CUR: getNewCursor(uri)

    Note over WC,CUR: READ OPERATION
    RU->>WC: cursor->search(key)
    WC->>CUR: cursor->search(key)
    CUR->>CUR: session = CUR2S(cbt)
    CUR->>BT: __wt_row_search(cbt, key)
    BT->>TX: __wt_txn_visible(session, txn_id)
    TX-->>BT: visible?
    BT-->>CUR: result
    CUR-->>WC: record
    WC-->>RU: Record
```

## Memory Layout

```mermaid
graph TB
    subgraph CONNECTION["WT_CONNECTION_IMPL"]
        direction TB
        S1[WT_SESSION_IMPL]
        S2[WT_SESSION_IMPL]
        S3[...cached sessions]
        BT1[WT_BTREE]
        BT2[WT_BTREE]
    end

    subgraph SESSION["WT_SESSION_IMPL"]
        direction LR
        TXN[WT_TXN]
        CUR_CACHE[cursor cache]
    end

    subgraph CURSOR["WT_CURSOR_BTREE"]
        direction LR
        IFACE[WT_CURSOR iface]
        REF[WT_REF* ref]
    end

    subgraph PAGE["WT_PAGE"]
        direction TB
        MOD[WT_MODIFY]
        UPDATES[WT_UPDATE chain]
    end

    CONNECTION --> SESSION
    SESSION --> CURSOR
    CURSOR --> PAGE

    style TXN fill:#ffcccc
    style CUR_CACHE fill:#ccccff
    style UPDATES fill:#ccffcc
```

## Visibility Check Path

```mermaid
graph TD
    START[cursor->next/search] --> CUR2S[CUR2S get session]
    CUR2S --> BT_OP[__wt_row_search/cursor operation]
    BT_OP --> PAGE_IN[__wt_page_in_func]
    PAGE_IN --> GET_UPDATES[get update chain from page]
    GET_UPDATES --> VISIBLE[__wt_txn_visible]

    subgraph VISIBILITY["Transaction Visibility Check"]
        VISIBLE --> CHECK_ID{txn_id visible?}
        CHECK_ID -->|yes| CHECK_TS{timestamp visible?}
        CHECK_ID -->|no| SKIP[skip this version]
        CHECK_TS -->|yes| RETURN[return this version]
        CHECK_TS -->|no| SKIP
    end

    SKIP --> NEXT_VERSION[next update in chain]
    NEXT_VERSION --> VISIBLE

    style VISIBLE fill:#ffcccc
    style RETURN fill:#ccffcc
    style SKIP fill:#cccccc
```
